In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Load the training and test data CSVs into pandas dataframes
trn_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv') #training 
tst_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv') #Test data
trn_df

In [ ]:
#Explore the dataset's text-based columns
trn_df.describe(include='object') #summary of a text (object) columns like text, keyword, location 

In [ ]:
# drop 'id' and 'target' from the training set (we will use them later)
trn = trn_df.drop(['id', 'target'], axis=1) #remove id and target from training data
tst = tst_df.drop(['id'], axis=1) #remove id from test data 

In [ ]:
# convet each row in the dataset to a python dictionary (easier for text processing)
trn_list = trn.to_dict(orient='records') # each row becomes a dictionary: ('text':..., 'keyword':..., ...)
tst_list = tst.to_dict(orient='records') # same for the test set
trn_list[:2]

In [ ]:
# convert each dictionry row to a string - the model will read the string
trn_txt = [str(row) for row in trn_list] # each dictionary is converted into a single string
tst_txt = [str(row) for row in tst_list] # same for test set
trn_txt[:2], tst_txt[:2] # show the first 2 converted strings from the both sets 

In [ ]:
# add the converted text as a new column called 'input'
trn_df['input'] = trn_txt #add a new column 'input' for training data
tst_df['input'] = tst_txt #same for test data
trn_df.input.head(), tst_df.input.head() #show first few values for new 'input' column

In [ ]:
# convert pandas Dataframes to HuggingFace Datasets (for use with Trainsformers)
from datasets import Dataset, DatasetDict #Import tools to create huggingface datasets
trn_ds = Dataset.from_pandas(trn_df) # convert training dataframe to hugging dataset
tst_ds = Dataset.from_pandas(tst_df) #convert test dataframe similarly
trn_ds, tst_ds #view datasets 

In [ ]:
# load tokenizer from pretrained DeBERTa model 
model_nm = 'microsoft/deberta-v3-small' #Name of the pretrained DeBERTa model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokz = AutoTokenizer.from_pretrained(model_nm) #load tokenizer for that model 

In [ ]:
#Tokenize the text using the tokenizer
def tok_func(x): return tokz(x["input"]) #A function to tokenize 'input' column

#Apply tokenizer to both datasets
trn_tok_ds = trn_ds.map(tok_func, batched=True) #Apply tokenizer to training dataset in batches
tst_tok_ds = tst_ds.map(tok_func, batched=True) #Apply tokenizer to training dataset in batches

In [ ]:
#rename 'target' column to 'labels' - required by huggingface models 
trn_tok_ds = trn_tok_ds.rename_columns({'target':'labels'}) #rename 'target' to 'labels'

In [ ]:
#split the training data into train and validation sets (99% train, 1%valid)
dds = trn_tok_ds.train_test_split(0.01, seed=42) #use 1% data for validation 
dds #show the split datasets

In [ ]:
from transformers import TrainingArguments,Trainer
# bs = 128  # out-of-mem
bs = 32
# epochs = 4
# epochs = 1  # LB 0.82592
# epochs = 2  # LB 0.82470
# epochs = 3  # LB 0.

epochs = 1  # LB 0.

lr = 8e-5

In [ ]:
#Define training configuration settings
from transformers import TrainingArguments

args = TrainingArguments(
    'outputs',            
    learning_rate=lr,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    eval_strategy="epoch", 
                         per_device_train_batch_size=bs, 
                         per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, 
                         weight_decay=0.01, 
                         report_to='none')

In [ ]:
# Define how the model should be evaluated — using F1 Score
from sklearn.metrics import f1_score

def my_metric(eval_pred):                           # Evaluation function for predictions
    return {'F1': f1_score(*eval_pred, zero_division=0)}  # F1 score (better for unbalanced classes)


In [ ]:
# Load the DeBERTa model for binary classification (2 classes: 0 or 1)
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)


In [ ]:
# Create the Trainer object to train and evaluate the model
from transformers import Trainer

trainer = Trainer(
    model,                           # The DeBERTa model
    args,                            # Training arguments we set earlier
    train_dataset=dds['train'],      # Training data
    eval_dataset=dds['test'],        # Validation data
    tokenizer=tokz                   # Tokenizer used (for decoding)
)


In [ ]:
# Start training the DeBERTa model
trainer.train()

In [ ]:
#optionaal
# Check performance on the validation set
trainer.evaluate()


In [ ]:
# Use the trained model to predict labels for the test set
preds = trainer.predict(tst_tok_ds)  # Runs the trained model on the test dataset
preds  # Displays the raw prediction output (logits + metadata)

In [ ]:
preds.predictions  # Shows the raw logits (2 numbers per row = [score for class 0, score for class 1])

In [ ]:
class_preds = np.argmax(preds.predictions, axis=-1)  # Takes the index of the larger value in each row
class_preds # Now you get a flat array like: [1, 1, 0, 1, ...]


In [ ]:
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')  # Load sample_submission.csv
sub # Show what it looks like (columns: id, target)


In [ ]:
sub['target'] = class_preds  # Replace all 0s in 'target' column with model predictions
sub                          # Preview the updated submission DataFrame

In [ ]:
name = os.path.basename(model_nm)  # Gets the model name string (e.g., 'microsoft/deberta-v3-small')

# Save two versions of the file
sub.to_csv(f'submission_{name}_ep{epochs}.csv', index=False)  # Fancy version with model+epoch name
sub.to_csv('submission.csv', index=False)                     # Standard version (for uploading to Kaggle)
